# ハンズオン2: 投資する銘柄を選ぶ

## 今回の問題

6つの株式があります。この中から**ちょうど3つ**を選んで投資します。

望みは2つ、しかも互いに相反します。

- できるだけ**儲けたい**
- できるだけ**値動きが激しくないようにしたい**

この2つを両立させる組み合わせを探すのが、**ポートフォリオ最適化**です。「ポートフォリオ」とは、持っている金融商品の組み合わせのことです。

### 前のハンズオンとの関係

前回のチーム分けとまったく同じ構造です。**0か1かを選ぶ、組み合わせの問題**だからです。問題の見た目は全然違うのに、数学的には同じ形になる。これは最適化を学ぶうえで非常に面白い点です。

### 知っておきたい3つの言葉

#### 期待収益（expected return）

平均的にどれくらい増えそうか、という見込みです。`0.10` なら「だいたい年10%くらい増えそう」という意味になります。

#### リスク

投資の世界で「リスク」は、**危険度**ではなく**値動きのブレの大きさ**を指します。大きく上がることも、大きく下がることもある、という不安定さのことです。

日常語の「リスク」とは意味が違うので注意してください。

#### 共分散（covariance）

2つの銘柄が、どれくらい**一緒に動くか**を表す数字です。

同じ業界の会社の株は、一緒に上がって一緒に下がりやすい傾向があります。そういう銘柄ばかり持っていると、悪いときに全部同時に下がってしまいます。

逆に、バラバラに動く銘柄を組み合わせれば、片方が下がってももう片方で支えられます。これが**分散投資**の数学的な根拠です。「卵を一つのカゴに盛るな」という格言そのものです。

### 重要な注意

**このデータは教材用に作った架空の数字です。投資の助言ではありません。** 実際の投資判断には使わないでください。

In [ ]:
import itertools
import math

assets = ["A", "B", "C", "D", "E", "F"]
expected_returns = [0.10, 0.08, 0.13, 0.07, 0.11, 0.09]
covariance = [
    [0.040, 0.012, 0.018, 0.006, 0.014, 0.010],
    [0.012, 0.025, 0.010, 0.008, 0.009, 0.011],
    [0.018, 0.010, 0.055, 0.004, 0.020, 0.008],
    [0.006, 0.008, 0.004, 0.020, 0.006, 0.007],
    [0.014, 0.009, 0.020, 0.006, 0.038, 0.012],
    [0.010, 0.011, 0.008, 0.007, 0.012, 0.030],
]
budget = 3
n_assets = len(assets)

## 1. 「良さ」をひとつの数字にまとめる

コンピュータに「良い組み合わせを選んで」と頼むには、**良し悪しを1つの数字で表す**必要があります。

### 変数の決め方

各銘柄について、

- `1` … 買う
- `0` … 買わない

とします。`(1, 0, 1, 0, 1, 0)` なら「A、C、Eの3つを買う」という意味です。

### 点数の作り方

次の3つを足し合わせます。**この点数は小さいほど良い**と決めます。

```
点数 ＝ リスク × こわがり度 － 儲け ＋ 罰金
```

#### リスク × こわがり度

「こわがり度」は数式では `λ`（ラムダ）と書き、コードでは `risk_aversion` です。

- `λ = 0` … リスクを全然気にしない。とにかく儲け重視
- `λ` が大きい … 安定志向。儲けが減ってもブレを避けたい

同じデータでも、この値を変えると選ばれる銘柄が変わります。つまり**正解は一つではなく、価値観によって変わる**ということです。

#### マイナス儲け

儲けは**多いほど良い**ものです。でも今は「点数が小さいほど良い」に統一したいので、引き算にします。

暖かさを最大にすることと、寒さを最小にすることが同じなのと同じ理屈です。

#### 罰金（ペナルティ）

「ちょうど3つ選ぶ」というルールを守らせるための仕掛けです。3つからズレた分を2乗して足します。

2乗するのは、4つ選んでも2つ選んでも**同じだけ罰する**ためです（`+1` も `-1` も2乗すれば `1`）。

### 次のセルがすること

2つの関数を用意します。

- `portfolio_metrics` … 選んだ銘柄を均等に買ったとき、収益とブレがどうなるかを計算
- `binary_objective` … 上で説明した点数（小さいほど良い）を計算

In [ ]:
def portfolio_metrics(bits):
    selected_count = sum(bits)
    if selected_count == 0:
        return 0.0, 0.0

    weights = [bit / selected_count for bit in bits]
    expected_return = sum(weight * value for weight, value in zip(weights, expected_returns))
    variance = sum(
        weights[i] * covariance[i][j] * weights[j]
        for i in range(n_assets)
        for j in range(n_assets)
    )
    return expected_return, variance

def binary_objective(bits, risk_aversion, penalty):
    raw_return = sum(expected_returns[i] * bits[i] for i in range(n_assets))
    raw_risk = sum(
        bits[i] * covariance[i][j] * bits[j]
        for i in range(n_assets)
        for j in range(n_assets)
    )
    constraint_violation = (sum(bits) - budget) ** 2
    return risk_aversion * raw_risk - raw_return + penalty * constraint_violation


## 2. 答え合わせ用に、全部試す

### なぜ全部試すのか

6銘柄の選び方は64通り（2の6乗）しかありません。この程度なら一瞬で全部調べられます。

ここで得られる答えは**完全に正しい正解**です。あとで賢い手法や量子的な手法を試したとき、「その答えは本当に良いのか」を判定する基準になります。

正解が分かっている小さな問題で練習する。これが新しい手法を学ぶときの鉄則です。

### 次のセルで注目してほしいこと

「こわがり度」を `0.0` から `5.0` まで変えて実行します。出力の変化を見比べてください。

- **こわがり度が小さいとき** … 儲けの大きい銘柄が選ばれ、`volatility`（ブレ）も大きい
- **こわがり度が大きいとき** … ブレの小さい銘柄が選ばれ、`expected_return`（儲け）も小さくなる

これは**トレードオフ**と呼ばれる関係です。片方を良くすると、もう片方が悪くなる。投資に限らず、現実のほとんどの意思決定にはこの構造があります。

### 出力の読み方

| 項目 | 意味 |
|---|---|
| `selected` | 選ばれた銘柄 |
| `expected_return` | 見込みの儲け（大きいほど良い） |
| `volatility` | 値動きのブレ（小さいほど安心） |
| `objective` | 点数（小さいほど良い） |
| `feasible` | ルール（3つ選ぶ）を守れたか |

「どれが唯一の正解か」ではなく、**どの価値観を選ぶか**という問題であることを実感してください。

In [ ]:
def solve(risk_aversion, penalty=10.0):
    candidates = itertools.product((0, 1), repeat=n_assets)
    objective, bits = min(
        (binary_objective(bits, risk_aversion, penalty), bits)
        for bits in candidates
    )
    selected = [asset for asset, bit in zip(assets, bits) if bit]
    expected_return, variance = portfolio_metrics(bits)
    return {
        "risk_aversion": risk_aversion,
        "selected": selected,
        "expected_return": expected_return,
        "volatility": math.sqrt(variance),
        "objective": objective,
        "feasible": sum(bits) == budget,
    }

for risk_aversion in (0.0, 0.5, 1.0, 2.0, 5.0):
    print(solve(risk_aversion))

## 3. 罰金はいくらにすればいい？

### 罰金が安すぎると、ルールは守られない

駐車違反の罰金が10円だったら、みんな平気で違反駐車をします。罰金より便利さのほうが大きいからです。

最適化でもまったく同じことが起きます。罰金が小さすぎると、コンピュータは**ルールを破ったほうが得だ**と判断してしまいます。

### 次のセルで起きること

ここでは「こわがり度 `0`」、つまり**儲けだけを追いかける**設定で試します。欲張りな状態なので、ルールを破る誘惑が最も強くなります。

結果を見てください。

- 罰金 `0.0` や `0.01` のとき … **6つ全部買ってしまう**（ルール違反）
- 罰金 `0.1` 以上のとき … ちゃんと3つに収まる

罰金が安いうちは「ルールを破ってでも全部買ったほうが点数が良い」と機械が判断していることが、はっきり分かります。

### では、高ければ高いほど良いのか

いいえ、そうではありません。罰金を極端に大きくすると、別の問題が起きます。

本来比べたい「儲けとリスクの微妙な差」が、巨大な罰金の数字に埋もれて見えなくなってしまうのです。1億円と1億100円を比べるようなもので、肝心の100円の差が誤差に見えてしまいます。

実際の量子マシンや最適化マシンは扱える数値の精度に限りがあるため、これは現実的な問題になります。

### 結論

> **ルールを守らせるのに十分な、できるだけ小さい罰金を探す**

この調整は実務で必ず必要になる作業です。

In [ ]:
for penalty in (0.0, 0.01, 0.1, 1.0, 10.0):
    candidates = itertools.product((0, 1), repeat=n_assets)
    objective, bits = min(
        (binary_objective(bits, risk_aversion=0.0, penalty=penalty), bits)
        for bits in candidates
    )
    print(f"penalty={penalty:>5}: selected={sum(bits)}, bits={bits}, objective={objective:.4f}")

## 4. 量子マシンへ渡すときに確認すること

この問題も、前回のチーム分けと同じく**QUBO**（0と1だけの、掛け算2つまでの式）の形をしています。つまり最適化マシンに渡せる形です。

渡す前に確認すべきことを挙げます。細かい話なので、「確認が必要な項目がある」という点だけ押さえれば十分です。

### 変換時の確認

1. `x × x = x` になること（0と1だけなので、2乗しても変わりません）
2. 共分散の表は対称なので、同じ組み合わせを**2回数えていないか**
3. 罰金の式を展開したときの各項が正しいか
4. 数字の大きさが、機械の扱える範囲に収まっているか

### 結果の評価

「答えが返ってきた」で終わらせず、次を見ます。

- **ルール違反の割合** … ちゃんと3つ選べた回数は何%か
- **正解との差** … 全部試した答えとどれだけ近いか
- **ばらつき** … 何度か実行したとき、毎回同じくらい良い答えが出るか

量子マシンの答えは毎回少しずつ違います。1回うまくいっただけでは評価できません。

### 使ってはいけないもの

終了したQIOサービス向けの古いサンプルコード（`azure.quantum.optimization` を使うもの）は動きません。インターネットで古い記事を見つけても、そのまま使わないでください。

## 実務ではもっと複雑になる

今回のモデルは、学習用にかなり単純化してあります。実際の運用では次のような要素が加わります。

### 買い方の制約

- **何株買うかを細かく決めたい**（今回は「買う・買わない」の2択でした）
- **最低売買単位がある**（1株単位で買えないことがあります）
- **売買するとき手数料がかかる**

### 投資方針による制約

- 「同じ業種に偏らない」
- 「特定の国への投資は全体の○%まで」
- 「環境や社会に配慮した企業を優先する」（ESG投資と呼ばれます）

### データそのものの不確かさ

ここが最も重要かもしれません。

**今回使った「期待収益」や「共分散」は、過去のデータから推定した値にすぎません。** 未来が過去と同じように動く保証はどこにもありません。

どれだけ精密に最適化しても、入力データが外れていれば結果も外れます。「ゴミを入れればゴミが出る」という、コンピュータ全般に当てはまる原則です。

### 計算量への影響

条件が増えるほど変数が増え、補助的な変数も必要になります。そのため量子マシンで扱う場合は、**量子ビットの数だけでなく**、回路の長さ、実行回数、古典コンピュータとの往復回数まで含めて考える必要があります。

## 演習

### 演習1: 選ぶ数を変える

`budget` を `2` や `4` に変えて実行してください。選ばれる銘柄と、儲け・ブレがどう変わるか観察しましょう。

**考えるポイント:** 銘柄数を増やすと、なぜブレが小さくなりやすいのでしょうか。「卵を一つのカゴに盛るな」を思い出してください。

### 演習2: データを変えてみる

銘柄Cの期待収益 `0.13` を `0.20` に変えて実行してください。

**考えるポイント:** Cは必ず選ばれるようになるでしょうか。選ばれないとしたら、それはなぜでしょうか。共分散の表のC行を見てみてください。

### 演習3: 新しいルールを追加する

「銘柄Aと銘柄Cは同時に選べない」という条件を加えてください。

**ヒント:** `binary_objective` に `penalty * bits[0] * bits[2]` を足します。両方が `1` のときだけ罰金が発生します。

### 演習4: 比較の物差しを考える

量子的な手法と普通の手法を**公平に**比べるには、どんな項目を測るべきでしょうか。5つ挙げてください。

**ヒント:** 「答えの良さ」以外にも、時間、費用、安定性など見るべきものがあります。